# AI Model Validation: From Mathematics to Trust
## Episode 16 — Consistency, Reproducibility, and Variance: Will the AI Give Us the Same Quality Again?

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

Episode 15 established that explanations themselves must be validated. Episode 16 asks a fundamental question for probabilistic AI, LLMs and agents:

> **If an AI system succeeds once, how much evidence do we have that it will succeed again?**

### One Successful Run ≠ Reproducible AI Quality

A single run is an observation. Trust requires evidence about the distribution of repeated behaviour.

### Task → Repeated Runs → Outcome Distribution → Variance → Agreement → Sensitivity → Reproducibility → Uncertainty → Evidence → Trust Decision


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys
rng=np.random.default_rng(1616)
print("Python:",sys.version.split()[0])
print("NumPy:",np.__version__)
print("System:",platform.system(),platform.machine())


## 1. One Run Is an Observation

Suppose one execution produces quality:

### Q₁=0.92

This does not tell us whether the next run will produce 0.92, 0.90, 0.50, or fail.

For repeated executions:

### Q₁,Q₂,...,Q_R

we can study a distribution.

### Observation ≠ Distribution


## 2. Consistency

Consistency asks how similar behaviour remains across comparable executions.

### Var(Q)=E[(Q−E[Q])²]

Low variance can mean consistent behaviour.

But a consistently incorrect model is still incorrect.

### Consistency ≠ Correctness


## 3. Repeatability

Repeatability asks whether sufficiently similar results occur when the same experiment is repeated under the same defined conditions.

Those conditions must be recorded rather than assumed.


## 4. Reproducibility

Reproducibility asks whether a result can be regenerated under a specified experimental and computational configuration.

That configuration may include:

### Model + Data + Prompt + Parameters + Seed + Runtime + Dependencies + Environment + External State

Terminology differs across scientific disciplines, so the operational definition should always be documented.


## 5. Deterministic vs Stochastic AI

For an idealized deterministic function:

### y=f(x)

the same input under the same computational state produces the same output.

For stochastic AI:

### Y ~ P(Y|x,θ,c)

where c can include decoding configuration, random state, retrieval, tools and environment.

Validation must characterize the output distribution.


## 6. Mean Repeated-Run Quality

For R runs:

### Q̄=(1/R)Σ_r Q_r

This estimates expected quality under the tested run-generating process.

### Mean Quality ≠ Guaranteed Quality


## 7. Run-to-Run Variance

Sample variance:

### s²=[1/(R−1)]Σ_r(Q_r−Q̄)²

Standard deviation:

### s=√s²

These quantify dispersion across repeated runs.


In [ ]:
R=1000
quality=np.clip(rng.normal(.86,.08,R),0,1)
print("Mean:",quality.mean())
print("SD:",quality.std(ddof=1))
print("5th percentile:",np.quantile(quality,.05))
print("Minimum:",quality.min())
plt.figure(figsize=(8,5))
plt.hist(quality,bins=35)
plt.xlabel("Quality score"); plt.ylabel("Run count")
plt.title("Repeated-Run Quality Distribution")
plt.grid(alpha=.2); plt.show()


## 8. Standard Error

If runs are approximately independent and identically distributed:

### SE(Q̄)=s/√R

Increasing R reduces uncertainty about the mean.

It does not reduce the system's intrinsic run-to-run variance.


## 9. Confidence Interval

A large-sample approximation is:

### Q̄ ± z·SE(Q̄)

This describes uncertainty in estimated expected quality—not the range of future individual runs.


## 10. Mean Uncertainty vs Run Variability

With many runs, the mean can be estimated precisely while individual outputs remain highly variable.

### Precise Mean ≠ Consistent System


In [ ]:
for r in [10,30,100,1000]:
    q=quality[:r]
    se=q.std(ddof=1)/np.sqrt(r)
    print(f"R={r:4d} mean={q.mean():.4f} 95%CI≈({q.mean()-1.96*se:.4f},{q.mean()+1.96*se:.4f}) SD={q.std(ddof=1):.4f}")


## 11. Coefficient of Variation

For positive metrics:

### CV=s/Q̄

CV expresses variability relative to the mean.

It is inappropriate or unstable when the mean is near zero or changes sign.


## 12. Lower-Tail Quality

Average quality can conceal bad executions.

Useful quantities include:

### Q₀.₀₅ = 5th percentile

and:

### P(Q<τ)

where τ is a minimum acceptable quality threshold.


## 13. Run Failure Probability

Define:

### F_r=1[Q_r<τ]

Then:

### p̂_fail=(1/R)Σ_rF_r

Consistency therefore connects directly to reliability.


In [ ]:
tau=.70
fail=quality<tau
print("Minimum acceptable quality:",tau)
print("Observed run failure rate:",fail.mean())


## 14. Same Mean, Different Variance

Two systems can have:

### E[Q_A]=E[Q_B]

while:

### Var(Q_A)≪Var(Q_B)

The average alone cannot distinguish them.


In [ ]:
A=np.clip(rng.normal(.85,.03,5000),0,1)
B=np.clip(rng.normal(.85,.14,5000),0,1)
for name,q in [("A",A),("B",B)]:
    print(name,"mean=",q.mean(),"SD=",q.std(ddof=1),"P(Q<.70)=",(q<.70).mean())
plt.figure(figsize=(8,5))
plt.hist(A,bins=50,alpha=.5,label="System A")
plt.hist(B,bins=50,alpha=.5,label="System B")
plt.xlabel("Quality"); plt.ylabel("Runs")
plt.title("Same Mean, Different Variance")
plt.legend(); plt.show()


## 15. Stochastic Generation

A generative model can sample tokens:

### X_t ~ P(X_t|X_<t,prompt)

Different random trajectories can produce different completions even when the prompt and model are unchanged.


## 16. Temperature

A common temperature transformation is:

### P_T(i)=exp(z_i/T)/Σ_jexp(z_j/T)

Lower temperature generally sharpens the distribution; higher temperature generally flattens it.

### Temperature ≠ Direct Measure of Creativity


In [ ]:
logits=np.array([3.,2.,1.,0.])
def softmax_temp(z,T):
    a=z/T; a-=a.max(); e=np.exp(a); return e/e.sum()
for T in [.3,.7,1.,1.5,2.]:
    print("T=",T,np.round(softmax_temp(logits,T),4))


## 17. Temperature and Variance

Changing the sampling distribution can change diversity and evaluation variance.

The effect on semantic quality is task- and model-dependent.

### Higher Temperature ≠ Automatically Worse
### Lower Temperature ≠ Guaranteed Determinism


## 18. Random Seeds

A seed controls a pseudo-random number stream in a specified implementation.

It can improve repeatability, but identical seeds do not guarantee identical outputs across every provider, runtime, hardware stack or external tool.

### Same Seed ≠ Universal Reproducibility


In [ ]:
def experiment(seed):
    r=np.random.default_rng(seed)
    return r.normal(size=5)
print(experiment(42))
print(experiment(42))
print(experiment(43))


## 19. Sources of Variance

Observed variance can originate from input sampling, training data, initialization, optimization, checkpoints, inference sampling, prompts, retrieval, tools, external APIs, environment and evaluation.

Validation should locate variance rather than reporting one undifferentiated number.


## 20. Law of Total Variance

For conditioning variable Z:

### Var(Y)=E[Var(Y|Z)]+Var(E[Y|Z])

This separates variation within conditions from variation between conditions.


## 21. Hierarchical Evaluation

For task i and repeated run r:

### Q_ir=μ+α_i+ε_ir

α_i can represent task difficulty.

ε_ir represents within-task run variation.

This distinguishes difficult tasks from unstable execution.


In [ ]:
tasks=100; runs=30
task_effect=rng.normal(0,.08,tasks)
Q=.82+task_effect[:,None]+rng.normal(0,.04,(tasks,runs))
print("Mean within-task variance:",np.mean(np.var(Q,axis=1,ddof=1)))
print("Between-task mean variance:",np.var(np.mean(Q,axis=1),ddof=1))
print("Total observed variance:",np.var(Q.ravel(),ddof=1))


## 22. Repeated Measures

Evaluate each task multiple times rather than once.

This enables direct estimation of:

### Within-Task Variance
### Between-Task Variance
### Failure Frequency
### Worst-Run Behaviour
### Agreement


## 23. Categorical Agreement

For repeated categorical decisions:

### P_o = observed agreement

Raw agreement can be misleading when one class dominates.


## 24. Cohen's Kappa

For two categorical result sets:

### κ=(P_o−P_e)/(1−P_e)

where P_e is expected chance agreement from the marginal frequencies.

Kappa should be interpreted with prevalence and marginal distributions.


In [ ]:
N=3000
latent=rng.normal(size=N)
r1=(latent+rng.normal(0,.6,N)>0).astype(int)
r2=(latent+rng.normal(0,.6,N)>0).astype(int)
po=np.mean(r1==r2)
p1,p2=r1.mean(),r2.mean()
pe=p1*p2+(1-p1)*(1-p2)
print("Observed agreement:",po)
print("Chance agreement:",pe)
print("Kappa:",(po-pe)/(1-pe))


## 25. Correlation Is Not Agreement

Two continuous outputs can be highly correlated while differing systematically.

### Correlation ≠ Agreement

Agreement analysis should inspect paired differences and bias.


## 26. Paired Difference Analysis

For paired outputs A_i and B_i:

### d_i=A_i−B_i

Inspect:

### d̄

and the dispersion of d_i.

A Bland–Altman-style analysis can reveal systematic offsets hidden by correlation.


In [ ]:
x=rng.normal(50,10,500)
y=x+8+rng.normal(0,2,500)
d=y-x
print("Correlation:",np.corrcoef(x,y)[0,1])
print("Mean paired difference:",d.mean())
print("SD difference:",d.std(ddof=1))
plt.figure(figsize=(8,5))
plt.scatter((x+y)/2,d,s=12,alpha=.4)
plt.axhline(d.mean())
plt.axhline(d.mean()+1.96*d.std(ddof=1),linestyle="--")
plt.axhline(d.mean()-1.96*d.std(ddof=1),linestyle="--")
plt.xlabel("Pair mean"); plt.ylabel("Difference")
plt.title("Agreement Through Paired Differences")
plt.grid(alpha=.2); plt.show()


## 27. Prompt Sensitivity

For semantically equivalent prompts p₁,...,p_K:

### S_prompt=SD_k[Q(p_k)]

This measures one form of prompt sensitivity.

### Same Intent ≠ Same Model Behaviour


## 28. Metamorphic Consistency

If transformation T should preserve the correct target:

### y(T(x))=y(x)

then test:

### d(f(T(x)),f(x))

Examples can include valid paraphrases, equivalent units, formatting changes or other justified invariances.


## 29. Agentic AI Variance

Agent systems introduce additional variance through planning, tool selection, retrieval, execution order, retries, intermediate state and stochastic generation.

### Agent Variance > Model-Call Variance can occur


## 30. Multi-Step Compounding

If k required steps independently succeed with probability r:

### P(workflow success)=r^k

Longer workflows can amplify small per-step failure probabilities.


In [ ]:
for r in [.95,.98,.99,.995]:
    print(f"Per-step reliability={r:.3f} -> 20-step workflow={r**20:.3%}")


## 31. Agent Path Variability

Across repeated agent executions record steps, tools, tokens, latency, cost, final quality and failure mode.

### Same Final Answer ≠ Same Operational Behaviour


## 32. Retrieval Variance

For RAG:

### Y~P(Y|X,R)

If retrieved context R changes, output variance may originate upstream.

Retrieval consistency and generation consistency should be measured separately.


## 33. Tool and External-State Variance

Agents can depend on changing APIs, databases, search results, clocks, sensors and files.

Reproducing an agent run may require preserving external state—not merely a seed.


## 34. Model Version Variance

A model update can produce:

### f_v1(x)≠f_v2(x)

even when the prompt and evaluation data remain unchanged.

Every evidence record therefore needs the model version.


## 35. Exact vs Statistical Reproducibility

**Exact reproducibility:** identical output.

**Statistical reproducibility:** repeated outputs remain sufficiently similar in distribution or continue satisfying acceptance criteria.

For stochastic AI, statistical reproducibility is often the more meaningful target.


## 36. Reproducibility Envelope

Define acceptable deviation:

### |M_run−M_ref|≤ε

or another task-specific equivalence criterion.

ε is a scientific and operational tolerance—not a universal constant.


## 37. Distributional Reproducibility

Compare repeated-run distributions using quantities such as mean difference, variance difference, KS distance, Wasserstein distance or Jensen–Shannon divergence.

Different metrics answer different questions.


## 38. Bootstrap Uncertainty of Variance

The observed standard deviation is itself an estimate.

Bootstrap repeated runs to obtain uncertainty intervals for mean, SD, failure rate or quantiles.


In [ ]:
q=quality[:200]
Bboot=2500
boot_sd=np.empty(Bboot)
for b in range(Bboot):
    sample=rng.choice(q,size=len(q),replace=True)
    boot_sd[b]=sample.std(ddof=1)
print("Observed SD:",q.std(ddof=1))
print("Bootstrap 95% interval:",np.quantile(boot_sd,[.025,.975]))


## 39. How Many Repeated Runs?

There is no universal repetition count.

Required runs depend on target quantity, variance, rare-failure probability, desired precision, confidence level, cost, dependence and decision stakes.


## 40. Precision Planning

Under a simple independent normal approximation, estimating a mean within margin m can use:

### R≈(zσ/m)²

Pilot evidence is needed when σ is unknown.


In [ ]:
sigma=.08
for margin in [.05,.02,.01,.005]:
    print(f"Margin ±{margin:.3f}: R≈{np.ceil((1.96*sigma/margin)**2):.0f}")


## 41. Rare Failures Require More Evidence

If zero failures occur in R approximately independent trials:

### p_fail,upper≈3/R

at roughly 95% confidence.

Rare-failure assurance can require far more repetitions than mean estimation.


## 42. Correlated Repetitions

Runs can share caches, sessions, retrieved documents, provider state or temporal conditions.

### R Runs ≠ R Independent Runs

Dependence reduces the effective amount of evidence.


## 43. Effective Sample Size

For a simplified stationary correlated sequence:

### R_eff≈R/[1+2Σ_kρ_k]

Positive autocorrelation can reduce effective sample size.

The formula relies on assumptions and should not be applied mechanically.


## 44. Environment Reproducibility

Relevant dimensions can include operating system, CPU/GPU, library version, model runtime, quantization, serving stack, region or provider.

Test the dimensions that materially affect the intended deployment.


## 45. Numerical Reproducibility

Finite-precision arithmetic and execution order can create numerical differences.

For many systems, decision equivalence is more meaningful than bitwise equality.

### Numerical Identity ≠ Always Required


## 46. Data Reproducibility

Preserve dataset identifier, version, split, sampling rule, preprocessing, label version and timestamp.

A result cannot be scientifically reconstructed if the evaluated data cannot be reconstructed.


## 47. Evaluator Variance

Observed score variability may include variability from the evaluation process itself.

### Observed Variance = System Variance + Evaluator Variance + Interaction

The evaluator is part of the measurement system.


## 48. Human-Rater Variance

Human evaluation should preserve disagreement and measure inter-rater reliability.

### Human Label ≠ Perfect Ground Truth


## 49. LLM-as-Judge Variance

An LLM judge can be stochastic, prompt-sensitive, position-sensitive and version-sensitive.

Validate judge repeatability, prompt sensitivity, position bias, agreement with humans and version stability.

### AI Evaluator ≠ Measurement Oracle


## 50. Variance Budget

A conceptual engineering decomposition is:

### V_total≈V_input+V_prompt+V_model+V_retrieval+V_tool+V_evaluator+Interactions

An additive statistical decomposition requires an appropriate experimental design.


## 51. Factorial Experiments

Systematically vary factors:

### Prompt × Temperature × Model Version × Retrieval Mode

Factorial experiments can reveal main effects and interactions more efficiently than ad hoc testing.


## 52. Interaction Effects

Prompt sensitivity may depend on model version. Temperature effects may depend on task. Retrieval effects may depend on query type.

### Total Behaviour ≠ Sum of Independent Effects


## 53. Subgroup Consistency

Episode 14 extends naturally:

### Var(Q|G=g)

### P(Failure|G=g)

### Agreement_g

Groups can have equal mean quality but unequal variance.

### Equal Mean ≠ Equal Consistency


## 54. Consistency Under Stress

Repeat each Episode 10 stress condition multiple times:

### Q_(stress,run)

This separates stress-induced degradation from ordinary stochastic variation.


## 55. Consistency Under Drift

Variance can change over time:

### Var(Q_t)

A rise in variance can be an investigation signal even before mean performance changes substantially.

It is not by itself proof of distribution drift.


## 56. Consistency and Calibration

A system can be calibrated on average while individual stochastic outputs remain unstable.

### Calibration ≠ Repeatability


## 57. Consistency and Safety

When low-quality tail runs have severe consequences, safety analysis should include:

### P(Q<τ)
### Tail Severity
### Variance Under Critical Scenarios

Average quality alone is insufficient.


## 58. Version Regression Testing

For a new model version, compare paired task-level performance:

### Δ_i=M_new,i−M_old,i

and compare repeated-run variance.

A slightly higher mean does not automatically justify deployment.


## 59. Reproducibility Manifest

Every experiment should preserve:

### Code Version
### Model Version
### Data Version
### Prompt Version
### Evaluator Version
### Parameters
### Seeds
### Dependencies
### Environment
### Timestamp
### External Resources

This turns a result into reconstructable evidence.


## 60. Multi-Dimensional Acceptance Gate

A consistency gate can require:

### Mean Quality ≥ μ_min
### SD ≤ σ_max
### P(Q<τ) ≤ p_max
### Agreement ≥ a_min
### Lower Quantile ≥ q_min

Thresholds must come from intended-use requirements.

### One Average Score ≠ Acceptance Evidence


In [ ]:
def gate(q):
    return {
        "mean":float(q.mean()),
        "sd":float(q.std(ddof=1)),
        "failure_rate":float(np.mean(q<.70)),
        "pass_mean":bool(q.mean()>=.85),
        "pass_sd":bool(q.std(ddof=1)<=.05),
        "pass_failure":bool(np.mean(q<.70)<=.01)
    }
print("System A:",gate(A))
print("System B:",gate(B))


## 61. Continuous Consistency Monitoring

Monitor over time:

### Mean(Q_t)
### SD(Q_t)
### Quantiles(Q_t)
### FailureRate_t
### Agreement_t
### PromptSensitivity_t
### PathVariance_t
### CostVariance_t
### LatencyVariance_t

Consistency is a time-varying property.


## 62. OpenVals Evidence Model

OpenVals should preserve task/data/model/prompt/evaluator versions, decoding configuration, seeds where meaningful, repeated-run outputs, mean, variance, tails, failure probability, confidence intervals, agreement, prompt sensitivity, retrieval/tool variance, agent path variance, evaluator variance, subgroup consistency, stress variance, environment manifest and acceptance criteria.

The stronger question is:

### How reliably can we reproduce acceptable behaviour, under which conditions, and with what uncertainty?


## 63. Scientific Consistency Chain

### Task → Repeated Runs → Outcome Distribution → Variance → Agreement → Sensitivity → Reproducibility → Uncertainty → Evidence → Trust Decision → Monitoring

A demonstration becomes evidence only when repeatability and uncertainty are measured.


## 64. Mathematical Summary

### Q̄=(1/R)Σ_rQ_r

### s²=[1/(R−1)]Σ_r(Q_r−Q̄)²

### SE(Q̄)=s/√R

### CV=s/Q̄

### F_r=1[Q_r<τ]

### p̂_fail=(1/R)Σ_rF_r

### P_T(i)=exp(z_i/T)/Σ_jexp(z_j/T)

### Var(Y)=E[Var(Y|Z)]+Var(E[Y|Z])

### κ=(P_o−P_e)/(1−P_e)

### S_prompt=SD_k[Q(p_k)]

### P(workflow success)=r^k

### R≈(zσ/m)²

### p_fail,upper≈3/R

### R_eff≈R/[1+2Σ_kρ_k]


## 65. Central Conclusion

### One Successful Run ≠ Reproducible AI Quality
### Mean Performance ≠ Consistency
### Same Seed ≠ Universal Reproducibility
### Correlation ≠ Agreement
### More Runs ≠ More Independent Evidence

> **A trustworthy AI system should not merely demonstrate that it can succeed. Validation should measure how often, how consistently, and under which conditions that success can be reproduced.**

The scientific questions are:

### What varies between runs?
### How large is the variance?
### What happens in the lower tail?
### Which factors create instability?
### Are outputs reproducible across prompts, environments, versions, retrieval and tools?
### Does the evaluator itself introduce variance?
### Is consistency preserved across subgroups and stress conditions?
### How many repetitions support the claim?
### Does reproducibility remain stable after deployment?

OpenVals principle:

> **A single result is an observation. Repeated, quantified and reproducible behaviour becomes evidence.**

## Next Episode

**Episode 17 — LLM Evaluation: How Do We Scientifically Measure Generative AI?**

Episode 17 can move directly into exact-match limitations, semantic similarity, factuality, groundedness, hallucination, rubric evaluation, pairwise preference, LLM-as-judge, judge bias, benchmark contamination, pass@k, task-specific evaluation and multi-metric evidence.
